In [ ]:
#Pasos para conectarse a Kafka cloud (Aiven)
"""
1. Crear servicio kafka en aiven (https://console.aiven.io/): kafka-up
2. Activar kafka_authentication_methods.sasl y auto_create_topics_enable en configuraciones avanzadas y habilitar Apache Kafka REST API en gestión de servicio
3. Crear tópico: temperatura-aulas, 2 particiones y 2 réplicas
4. En Connection information, selecciona el método SASL: anota el usuario (avnadmin), la contraseña, y descarga el archivo CA Certificate (ca.pem)
5. Instala kafka-python
"""

flujo_laboratorio_kafka_spark.svg

In [ ]:
!pip install kafka-python
from kafka import KafkaProducer # libreria cliente de Kafka para Python

In [ ]:
%%writefile productor_sensores.py
import json
import os
import random
import time
from datetime import datetime
from kafka import KafkaProducer

# -----------------------------------------------------------------------------
#  1) DATOS DE CONEXION AL CLUSTER DE KAFKA EN AIVEN
# -----------------------------------------------------------------------------
#  Las credenciales NO se escriben en el codigo: se leen de variables de
#  entorno (archivo .env / entorno del proceso). Asi nunca se suben a un repo.
#  Copia los valores desde la consola de Aiven ("Connect information" ->
#  "Apache Kafka") a tu .env:
#     KAFKA_HOST, KAFKA_USER, KAFKA_PASSWORD, KAFKA_CA, KAFKA_TOPIC
KAFKA_HOST = os.getenv("KAFKA_HOST", "kafka-<id>.aivencloud.com:<puerto>")
KAFKA_USER = os.getenv("KAFKA_USER", "avnadmin")
KAFKA_PASSWORD = os.getenv("KAFKA_PASSWORD")  # requerido (no se hardcodea)
CA_FILE = os.getenv("KAFKA_CA", "ca.pem")
TOPIC = os.getenv("KAFKA_TOPIC", "temperatura-aulas")

if not KAFKA_PASSWORD:
    raise SystemExit(
        "Falta KAFKA_PASSWORD. Definelo en el entorno (.env) antes de ejecutar; "
        "nunca se escribe en el codigo."
    )


# -----------------------------------------------------------------------------
#  2) CREACION DEL PRODUCTOR
# -----------------------------------------------------------------------------
#  bootstrap_servers  : a que broker conectarse para "entrar" al cluster.
#  security_protocol  : "SASL_SSL" = autenticacion (SASL) + cifrado (SSL).
#  sasl_mechanism     : algoritmo de autenticacion. Aiven usa "SCRAM-SHA-256".
#  sasl_plain_username/password : credenciales (desde el entorno).
#  ssl_cafile         : ruta al certificado CA, para validar al broker.
#  value_serializer   : convierte el VALOR del mensaje (dict) a bytes JSON.
producer = KafkaProducer(
    bootstrap_servers=KAFKA_HOST,
    security_protocol="SASL_SSL",
    sasl_mechanism="SCRAM-SHA-256",
    sasl_plain_username=KAFKA_USER,
    sasl_plain_password=KAFKA_PASSWORD,
    ssl_cafile=CA_FILE,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

# -----------------------------------------------------------------------------
#  3) GENERACION Y ENVIO DE LECTURAS
# -----------------------------------------------------------------------------
#  Lista de aulas "monitoreadas". En cada iteracion se elige una al azar.
aulas = ["A-101", "A-102", "B-201", "B-202", "LAB-INF"]

print("Enviando lecturas a Kafka... (Ctrl+C para detener)")
try:
    while True:
        # Se arma una lectura simulada (un diccionario de Python):
        #   aula, temperatura (18-32), humedad (30-80), timestamp ISO.
        lectura = {
            "aula": random.choice(aulas),
            "temperatura": round(random.uniform(18.0, 32.0), 2),
            "humedad": round(random.uniform(30.0, 80.0), 1),
            "timestamp": datetime.now().isoformat()
        }

        # producer.send() = PUBLICAR el mensaje en el topic (se envia en lotes).
        producer.send(TOPIC, value=lectura)
        print("Enviado:", lectura)

        # Pausa de 2 segundos: simula que el sensor mide cada 2 segundos.
        time.sleep(2)

except KeyboardInterrupt:
    print("\nDetenido por el usuario.")

finally:
    # flush() fuerza el envio del buffer; close() cierra la conexion ordenada.
    producer.flush()
    producer.close()

In [ ]:
!python productor_sensores.py